# Brief 02 : SCD1, SCD2, SCD3  corrigé

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")
import pandas as pd
import psycopg2

DATABASE_URL = "dbname=vetprice"
conn = psycopg2.connect(DATABASE_URL)
conn.autocommit = False
cur = conn.cursor()

In [2]:
ph = pd.read_sql("SELECT * FROM produit_historise", conn)
ph["price"] = pd.to_numeric(ph["price"])
print(ph.shape)

(331056, 11)


### A1

In [4]:
# --- SQL ---
display(pd.read_sql("""
    SELECT (SELECT count(*) FROM produit_historise) AS versions,
           (SELECT count(*) FROM (SELECT DISTINCT site, cle FROM produit_historise) x) AS produits
""", conn))

# --- pandas ---
print(
    len(ph), 
    "versions |", 
    ph.groupby(["site", "cle"]).ngroups, 
    "produits distincts"
)

,versions,produits
0,331056,140947


331056 versions | 140947 produits distincts


### A2

In [5]:
# --- SQL ---
scd1_sql = pd.read_sql("""
    SELECT site, cle, name, price
    FROM produit_historise WHERE is_current
    ORDER BY site, cle
""", conn)
print(len(scd1_sql), "lignes")
display(scd1_sql.head())

# --- pandas ---
scd1 = ph[ph.is_current][["site", "cle", "name", "price"]].sort_values(["site", "cle"])
print(len(scd1), "lignes")
display(scd1.head())

140947 lignes


,site,cle,name,price
0,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97
1,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95
2,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59
3,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69
4,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99


140947 lignes


,site,cle,name,price
1,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97
4,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95
6,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59
8,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69
10,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99


### A3

In [7]:
# --- SQL ---
scd3_sql = pd.read_sql("""
    WITH v AS (
        SELECT site, cle, name, price, is_current,
               lag(price) OVER (PARTITION BY site, cle ORDER BY valid_from) AS price_precedent
        FROM produit_historise
    )
    SELECT site, cle, name, price, price_precedent FROM v WHERE is_current
""", conn)
print(len(scd3_sql), "lignes,", int(scd3_sql.price_precedent.notna().sum()), "avec un prix précédent")
display(scd3_sql.head())

# --- pandas ---
v = ph.sort_values(["site", "cle", "valid_from"]).copy()
v["price_precedent"] = v.groupby(["site", "cle"])["price"].shift(1)
scd3 = v[v.is_current][["site", "cle", "name", "price", "price_precedent"]]
print(len(scd3), "lignes,", int(scd3.price_precedent.notna().sum()), "avec un prix précédent")
display(scd3.head())

140947 lignes, 102621 avec un prix précédent


,site,cle,name,price,price_precedent
0,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97,4.97
1,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95,31.95
2,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59,1.59
3,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69,11.69
4,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99,27.99


140947 lignes, 102621 avec un prix précédent


,site,cle,name,price,price_precedent
1,animalis,0000000856126,Animalis - Friandises au Fromage Souflé pour C...,4.97,4.97
4,animalis,0000003397510,Foolee - Brosse pour Chien de Moyenne Race - M,31.95,31.95
6,animalis,0000008384164,Edgard & Cooper - Barquette au Poulet et Saumo...,1.59,1.59
8,animalis,0000116018609,Seachem - PhosGuard Contrôle du Phosphate/Sili...,11.69,11.69
10,animalis,0000116020602,Seachem - CupriSorb Adsorbant de Cuivre et Mét...,27.99,27.99


### A4

on peut utiliser `UNION ALL`

In [8]:
# --- SQL ---
display(pd.read_sql("""
    SELECT 'SCD1' AS type, count(*) AS lignes FROM produit_historise WHERE is_current
    UNION ALL SELECT 'SCD2', count(*) FROM produit_historise
    UNION ALL SELECT 'SCD3', count(*) FROM produit_historise WHERE is_current
""", conn))

# --- pandas ---
courant = int(ph.is_current.sum())
display(pd.DataFrame({"type": ["SCD1", "SCD2", "SCD3"],
                      "lignes": [courant, len(ph), courant]}))

# SCD1 et SCD3 ont le meme nombre de lignes : une par produit.
# SCD3 coute une colonne de plus, pas une ligne de plus.
# SCD2 est environ 2,3 fois plus volumineux : c'est le prix de l'historique complet.

,type,lignes
0,SCD1,140947
1,SCD2,331056
2,SCD3,140947


,type,lignes
0,SCD1,140947
1,SCD2,331056
2,SCD3,140947


### A5

In [9]:
# --- SQL ---
display(pd.read_sql("""
    SELECT count(*) AS versions_grain_changement FROM (
        SELECT price, lag(price) OVER (PARTITION BY site, cle ORDER BY valid_from) AS p
        FROM produit_historise
    ) x WHERE p IS NULL OR price IS DISTINCT FROM p
""", conn))

# --- pandas ---
v = ph.sort_values(["site", "cle", "valid_from"]).copy()
v["p"] = v.groupby(["site", "cle"])["price"].shift(1)
identique = (v.price == v.p) | (v.price.isna() & v.p.isna())
garde = v.p.isna() | ~identique   # equivalent de : p IS NULL OR price IS DISTINCT FROM p
n_chg = int(garde.sum())
print(n_chg, "versions en grain changement contre", len(ph), "en grain observation")
print("ecart :", len(ph) - n_chg, "versions, soit", f"{100 * (len(ph) - n_chg) / len(ph):.1f}% de moins")

,versions_grain_changement
0,211023


211023 versions en grain changement contre 331056 en grain observation
ecart : 120033 versions, soit 36.3% de moins


### A6

**SCD1.** Il ne reste que l'état courant, donc une ligne par produit. Toutes le sinformations des versions précédentes sont perdues : l'historique d'un produit (Q4), le nombre de versions (Q5), le prix à une date donnée (Q6), le catalogue à une date passée (Q7), les variations de prix (Q8 et Q9). On ne peut répondre qu'aux questions sur qui concerne le présent (i.e. l'état courant) : Q1, Q2, Q3, et le matching inter sites Q10.

**SCD3.** On récupère une seule variation, la dernière : on peut dire de combien le prix a bougé au dernier changement, donc on peut répondre partiellement à Q9. On ne peut pas avoir accès à des infos plus anciennes et encore moins reconstituer un état à une date arbitraire (Q6, Q7 deviennent impossibles).

**Le grain.** Le grain "une version par observation" est préférable quand on veut prouver que le produit a bien été observé à chaque run : il trace la collecte, et permet de détecter qu'un produit a cessé d'apparaître. Le grain "une version par changement" est préférable pour la volumétrie et pour compter les vraies variations de prix, sans lignes redondantes. (ici l'écart est de 120 033 versions, soit 36.3 % de moins)